# Prepare Calendar Table

Generates and persists a complete calendar dimension table `adb_caresync.silver.calendar` covering **1940-01-01 → 2030-12-31**, with all standard date attributes for analytics and reporting.

In [0]:
from pyspark.sql import functions as F

# ── 1. Generate date spine: 1940-01-01 → 2030-12-31 ─────────────────────────
df = spark.sql(
    "SELECT explode(sequence(date'1940-01-01', date'2030-12-31', interval 1 day)) AS date"
)

# ── 2. Derive all calendar attributes ────────────────────────────────────────
calendar_df = df.select(

    # ── Surrogate key ──────────────────────────────────────────────────────
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),

    # ── Date ───────────────────────────────────────────────────────────────
    F.col("date"),

    # ── Year ───────────────────────────────────────────────────────────────
    F.year("date").alias("year"),
    F.trunc("date", "yyyy").alias("year_start_date"),
    F.make_date(F.year("date"), F.lit(12), F.lit(31)).alias("year_end_date"),

    # ── Quarter ────────────────────────────────────────────────────────────
    F.quarter("date").alias("quarter"),
    F.concat(
        F.year("date").cast("string"), F.lit("-Q"), F.quarter("date").cast("string")
    ).alias("year_quarter"),
    # Quarter start: first day of the first month of the quarter
    F.make_date(
        F.year("date"),
        (F.quarter("date") - 1) * 3 + 1,
        F.lit(1)
    ).alias("quarter_start_date"),
    # Quarter end: last day of the third month of the quarter
    F.last_day(
        F.make_date(F.year("date"), F.quarter("date") * 3, F.lit(1))
    ).alias("quarter_end_date"),

    # ── Month ──────────────────────────────────────────────────────────────
    F.month("date").alias("month"),
    F.date_format("date", "yyyyMM").cast("int").alias("year_month"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "MMM").alias("month_name_short"),
    F.trunc("date", "MM").alias("month_start_date"),
    F.last_day("date").alias("month_end_date"),

    # ── Week ───────────────────────────────────────────────────────────────
    F.weekofyear("date").alias("week_of_year"),   # ISO: week containing 1st Thursday

    # ── Day ────────────────────────────────────────────────────────────────
    F.dayofmonth("date").alias("day_of_month"),
    F.dayofweek("date").alias("day_of_week"),     # 1 = Sunday … 7 = Saturday
    F.dayofyear("date").alias("day_of_year"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.date_format("date", "EEE").alias("day_name_short"),

    # ── Flags ──────────────────────────────────────────────────────────────
    F.when(F.dayofweek("date").isin([1, 7]), F.lit(True))
     .otherwise(F.lit(False)).alias("is_weekend"),

    F.when(F.dayofweek("date").isin([1, 7]), F.lit(False))
     .otherwise(F.lit(True)).alias("is_weekday"),

    F.when(F.month("date").isin([3, 4, 5]), F.lit("Spring"))
     .when(F.month("date").isin([6, 7, 8]), F.lit("Summer"))
     .when(F.month("date").isin([9, 10, 11]), F.lit("Autumn"))
     .otherwise(F.lit("Winter")).alias("season"),
)

# ── 3. Write to Delta table ───────────────────────────────────────────────────
(
    calendar_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("adb_caresync.silver.calendar")
)

row_count = spark.table("adb_caresync.silver.calendar").count()
print(f"✅ adb_caresync.silver.calendar created with {row_count:,} rows")
display(spark.table("adb_caresync.silver.calendar").limit(10))